<a href="https://colab.research.google.com/github/vad-source/NLPAPP/blob/main/QA/DeepLearning_Based_Extractive_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## NLP APPLICATIONS
**Designed by:** RAJA VADHANA PRABHAKAR  
**Organization:** BITS PILANI WILP  
**Purpose:** Academic Training / Proof of Concept  

---
#### Attribution & AI Disclosure
- **Original Design:** The logic, architecture, and modular structure of this notebook were designed by the author.
- **Development Assistance:** Generative AI (e.g., ChatGPT/Claude/Copilot) was used for coding implementation and debugging support.
- **License:** This work is licensed under the [Apache License 2.0](https://apache.org).

In [ ]:
!pip -q install transformers torch pandas

In [ ]:
import torch
import pandas as pd

from transformers import pipeline
from transformers import AutoTokenizer
from transformers import AutoModelForQuestionAnswering

## 0. Knowledge Base

In [ ]:
documents = [{"id": 1,"context": """Patient was admitted with severe fever and cough.During hospitalization the patient received Penicillin and later developed a severe rash. The rash improved after discontinuation of Penicillin."""},
             {"id": 2,"context": """Patient has a history of diabetes mellitus.Metformin was prescribed for glucose control.No allergic reaction was observed."""}
]

qa_test = [{"question":"Which medication caused the rash?","context_id": 1,"ground_truth":"Penicillin"},
           {"question":"What disease does the patient have?","context_id": 2,"ground_truth":"diabetes mellitus"}
]


## 1. Query Processor

In [ ]:
#Learners may replace this with sophisticated techiques like GEC Question restructuting etc.,
class QueryProcessor:
    def process(self, question):
        return question.strip()


In [ ]:
query_processorT = QueryProcessor()
context_idT = 1
processed_questionT = query_processorT.process(qa_test[context_idT]["question"])
print("Sample Query : ", qa_test[context_idT]["question"])
print("Processed Query: ", processed_questionT)

Sample Query :  What disease does the patient have?
Processed Query:  What disease does the patient have?


In [ ]:
#Learners may replace this with sophisticated techiques like automated intent/context extraction
class DocumentProcessor:
    def get_context(self, context_id):
        for doc in documents:
            if doc["id"] == context_id:
                return doc["context"]
        return ""


## 2. Candidate Retriever

In [ ]:
class BertExtractiveQA:
    def __init__(self):
        print("Loading BERT QA model...")
        model_name ="distilbert-base-cased-distilled-squad"
        self.qa_pipeline = pipeline("question-answering",model=model_name,tokenizer=model_name)

    def predict(self, question, context):
        result = self.qa_pipeline(question=question,context=context)
        return result



In [ ]:
doc_processorT = DocumentProcessor()
bert_qaT = BertExtractiveQA()

contextT = doc_processorT.get_context(context_idT)
predictionT = bert_qaT.predict(processed_questionT,contextT)
candidate_answerT = predictionT["answer"]

print("Input Processed Query to BERT",processed_questionT)
print("Span Predictions from BERT",predictionT)
print("Final Answer",candidate_answerT)

Loading BERT QA model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Input Processed Query to BERT What disease does the patient have?
Span Predictions from BERT {'score': 0.6403404474258423, 'start': 93, 'end': 103, 'answer': 'Penicillin'}
Final Answer Penicillin


## 3. Answer Generator

In [ ]:
class PostProcessor:
    def curate(self, prediction):
        answer = prediction["answer"]
        score = prediction["score"]
        return answer.strip(), float(score)

In [ ]:
postprocessorT = PostProcessor()
final_answerT, confidenceT = postprocessorT.curate(predictionT)
print("Final Answer:", final_answerT)
print("Confidence:",round(confidenceT, 3))

Final Answer: Penicillin
Confidence: 0.64


## 4. Evaluator

In [ ]:
class Evaluator:
    def exact_match(self, pred, gt):
        return int(pred.lower() == gt.lower())

    def token_f1(self, pred, gt):
        pred_tokens = pred.lower().split()
        gt_tokens = gt.lower().split()
        common = set(pred_tokens) & set(gt_tokens)
        if len(common) == 0:
            return 0
        precision = len(common) / len(pred_tokens)
        recall = len(common) / len(gt_tokens)
        f1 = (2 * precision * recall) / (precision + recall)
        return round(f1, 3)

    def confidence_score(self, confidence):
        return round(confidence, 3)

    def aggregate_score(self, em, f1, conf):
        return round((em + f1 + conf) / 3,3)

##

## Pipeline

In [ ]:
print("=" * 60)
print("BERT EXTRACTIVE QA SYSTEM")
print("=" * 60)

query_processor = QueryProcessor()
doc_processor = DocumentProcessor()
bert_qa = BertExtractiveQA()
postprocessor = PostProcessor()
evaluator = Evaluator()

all_scores = []
results = []

for sample in qa_test:
    print("\n" + "=" * 60)
    question = sample["question"]
    context_id = sample["context_id"]
    gt = sample["ground_truth"]
    print("\nQUESTION:")
    print(question)
    print("\n[PHASE 1] QUERY PROCESSING")
    processed_question = query_processor.process(question)
    print("\n[PHASE 2] DOCUMENT PROCESSING")
    context = doc_processor.get_context(context_id)
    print("\nCONTEXT:")
    print(context)
    print("\n[PHASE 3] BERT SPAN PREDICTION")
    prediction = bert_qa.predict(processed_question,context)
    print("\nRAW MODEL OUTPUT:")
    print(prediction)
    print("\n[PHASE 4] CANDIDATE GENERATION")
    candidate_answer = prediction["answer"]
    print("\n[PHASE 5] POSTPROCESSING")
    final_answer, confidence = postprocessor.curate(prediction)
    print("Final Answer:", final_answer)
    print("Confidence:",round(confidence, 3))
    print("\n[PHASE 6] EVALUATION")
    em = evaluator.exact_match(final_answer, gt)
    f1 = evaluator.token_f1(final_answer,gt)
    conf = evaluator.confidence_score(confidence)
    final_score = evaluator.aggregate_score(em,f1,conf)
    all_scores.append(final_score)
    print("\nGROUND TRUTH:")
    print(gt)
    print("\nMETRICS")
    print("Exact Match:", em)
    print("Token F1:", f1)
    print("Confidence:", conf)
    print("Aggregate Score:",final_score)
    results.append({"Question": question,"Predicted": final_answer,"Ground Truth": gt,"Confidence": round(confidence, 3),"Exact Match": em,"Token F1": f1,"Final Score": final_score})

print("\n" + "=" * 60)
print("FINAL AGGREGATE RESULTS")
print("=" * 60)

avg_score = sum(all_scores) / len(all_scores)
print("\nAverage QA Score:",round(avg_score, 3))

df = pd.DataFrame(results)
print("\nDETAILED RESULTS")
display(df)

BERT EXTRACTIVE QA SYSTEM
Loading BERT QA model...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]



QUESTION:
Which medication caused the rash?

[PHASE 1] QUERY PROCESSING

[PHASE 2] DOCUMENT PROCESSING

CONTEXT:
Patient was admitted with severe fever and cough.During hospitalization the patient received Penicillin and later developed a severe rash. The rash improved after discontinuation of Penicillin.

[PHASE 3] BERT SPAN PREDICTION

RAW MODEL OUTPUT:
{'score': 0.8831359711948608, 'start': 93, 'end': 103, 'answer': 'Penicillin'}

[PHASE 4] CANDIDATE GENERATION

[PHASE 5] POSTPROCESSING
Final Answer: Penicillin
Confidence: 0.883

[PHASE 6] EVALUATION

GROUND TRUTH:
Penicillin

METRICS
Exact Match: 1
Token F1: 1.0
Confidence: 0.883
Aggregate Score: 0.961


QUESTION:
What disease does the patient have?

[PHASE 1] QUERY PROCESSING

[PHASE 2] DOCUMENT PROCESSING

CONTEXT:
Patient has a history of diabetes mellitus.Metformin was prescribed for glucose control.No allergic reaction was observed.

[PHASE 3] BERT SPAN PREDICTION

RAW MODEL OUTPUT:
{'score': 0.954978346824646, 'start': 25, 

,Question,Predicted,Ground Truth,Confidence,Exact Match,Token F1,Final Score
0,Which medication caused the rash?,Penicillin,Penicillin,0.883,1,1.0,0.961
1,What disease does the patient have?,diabetes mellitus,diabetes mellitus,0.955,1,1.0,0.985


In [ ]:
#This is a helper code added for training to illustrate the internal working of sub word tokenization performed and span probabilities detected by BERT.
model_name = "distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

question = "Which medication caused the rash?"

context = """
Patient was admitted with severe fever and cough.
During hospitalization the patient received
Penicillin and later developed a severe rash.
The rash improved after discontinuation of Penicillin.
"""
inputs = tokenizer(question, context, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

start_logits = outputs.start_logits[0]
end_logits = outputs.end_logits[0]
start_probs = torch.softmax(start_logits,dim=0)
end_probs = torch.softmax(end_logits, dim=0)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
rows = []
for token, s_prob, e_prob in zip(tokens,start_probs,end_probs):
    rows.append({"Token": token,"P(Start)": round(float(s_prob),6),"P(End)": round(float(e_prob),6)})
df = pd.DataFrame(rows)
display(df)

start_idx = torch.argmax(start_probs)
end_idx = torch.argmax(end_probs)
answer_tokens = tokens[start_idx : end_idx + 1]
answer = tokenizer.convert_tokens_to_string(answer_tokens)
print("\n" + "=" * 60)
print("BEST START TOKEN INDEX:",int(start_idx))
print("BEST END TOKEN INDEX:",int(end_idx))
print("\nFINAL ANSWER:")
print(answer)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

,Token,P(Start),P(End)
0,[CLS],0.000003,0.000010
1,Which,0.000001,0.000000
2,medication,0.000001,0.000003
3,caused,0.000000,0.000000
4,the,0.000000,0.000000
5,r,0.000000,0.000000
6,##ash,0.000000,0.000000
7,?,0.000002,0.000001
8,[SEP],0.000001,0.000001
9,Pat,0.000079,0.000000



BEST START TOKEN INDEX: 25
BEST END TOKEN INDEX: 27

FINAL ANSWER:
Penicillin
